In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
comments = pd.read_csv(r"C:\Users\akash\OneDrive\Desktop\UScomments.csv" , on_bad_lines="skip")

In [ ]:
comments.head(5)

In [ ]:
type(comments)

In [ ]:
comments.isnull().sum()

In [ ]:
comments.dropna(inplace= True)

In [ ]:
comments.isnull().sum()

In [ ]:
import nltk

In [ ]:
nltk.download("vader_lexicon")

In [ ]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer

In [ ]:
sia=SentimentIntensityAnalyzer()

In [ ]:
comments["comment_text"]

In [ ]:
sia.polarity_scores("MY FAN . attendance")

In [ ]:
sia.polarity_scores("MY FAN . attendance")['compound']

In [ ]:
sample_df=comments[0:10000]

In [ ]:
sentimen_scores =[]
for comment in comments["comment_text"]:
    score = sia.polarity_scores(str(comment))['compound']
    sentimen_scores.append(score)

In [ ]:
sample_df=comments[0:10000]

In [ ]:
 sentimen_scores

In [ ]:
comments["polarity"] = sentimen_scores

In [ ]:
comments.head(6)

In [ ]:
filter_pos=(comments["polarity"]>=0.8) & ((comments["polarity"] <=1.0))

In [ ]:
comments[filter_pos]

In [ ]:
comments_positive = comments[filter_pos]

In [ ]:
comments_positive.shape

In [ ]:
filter_neg=(comments["polarity"]>=-1.0) & ((comments["polarity"] <=-0.80))

In [ ]:
comments[filter_neg]

In [ ]:
comments_negative = comments[filter_neg]

In [ ]:
comments_negative.shape

In [ ]:
!pip install wordcloud

In [ ]:
type(comments_positive["comment_text"])

In [ ]:
total_positive_comments=' '.join((comments_positive["comment_text"]))

In [ ]:
total_positive_comments[0:1000]

In [ ]:
from wordcloud import WordCloud , STOPWORDS

In [ ]:
wordcloud_positive = WordCloud(stopwords=set(STOPWORDS)).generate(total_positive_comments)

In [ ]:
plt.imshow(wordcloud_positive)
plt.axis("off")

In [ ]:
total_negative_comments=' '.join((comments_negative["comment_text"]))

In [ ]:
wordcloud_negative = WordCloud(stopwords=set(STOPWORDS)).generate(total_negative_comments)

In [ ]:
plt.imshow(wordcloud_negative)
plt.axis("off")

In [ ]:
!pip install emoji==2.14.1

In [ ]:
import emoji

In [ ]:
comments.head(6)

In [ ]:
emoji_info = emoji.emoji_list("trending 😉")

In [ ]:
emoji_info

In [ ]:
[item["emoji"]for item in emoji_info]

In [ ]:
comments["comment_text"]

In [ ]:
all_emojis_found = []

for comment in comments["comment_text"]:
    emojis_info=emoji.emoji_list(comment)
    emojis_found=[item["emoji"]for item in emojis_info]
    all_emojis_found.extend(emojis_found)

In [ ]:
all_emojis_found[0:10]

In [ ]:
len(all_emojis_found)

In [ ]:
from collections import Counter


In [ ]:
emojis_count_list_top10 = Counter(all_emojis_found).most_common(10)

In [ ]:
emojis_count_list_top10

In [ ]:
emojis = [emoji for emoji , count in emojis_count_list_top10]
counts = [count for emoji , count in emojis_count_list_top10]

In [ ]:
emojis

In [ ]:
counts

In [ ]:
!pip install plotly

In [ ]:
import plotly.graph_objs as go 
from plotly.offline import iplot 

In [ ]:
import plotly.express as px
px.bar(x=emojis,
       y= counts,
      title= "Most used emojis in youtube comments" ,
      labels=
       {
           "x" : "Emoji",
           "y" : "count" ,
       })

In [ ]:
4.COLLECTION  OF ENTIRE DATA

In [ ]:
import os

In [ ]:
files = os.listdir(r'C:\Users\akash\Downloads\additional_data')

In [ ]:
files

In [ ]:
files_csv=[file for file in files if ".csv" in file]

In [ ]:
files_csv

In [ ]:
full_df = pd.DataFrame()

path = r'C:\Users\akash\Downloads\additional_data'
for file in files_csv:
    current_df = pd.read_csv(path+'/'+file, encoding="iso=8859-1" , on_bad_lines="skip")
    full_df = pd.concat([current_df,full_df], ignore_index = True)
    

In [ ]:
full_df.shape

In [ ]:
full_df.head(4)

In [ ]:
full_df.columns

In [ ]:
How to export your data to CSV, JSON & Databases !

In [ ]:
full_df[full_df.duplicated()].shape

In [ ]:
full_df = full_df.drop_duplicates()

In [ ]:
full_df.shape

In [ ]:
full_df[0:5000].to_csv(r'C:\Users\akash\Downloads\exported_data/Youtube_whole_data.csv' , index= False)

In [ ]:
full_df[0:5000].to_json(r'C:\Users\akash\Downloads\exported_data/Youtube_whole_data.json')

In [ ]:
!pip install sqlalchemy 

In [ ]:
from sqlalchemy import create_engine

In [ ]:
engine = create_engine(r'sqlite:///C:\Users\akash\Downloads\exported_data/youtube_data.sqlite')

In [ ]:
full_df.to_sql("Users" , con = engine,if_exists = "replace")

In [ ]:
Which category dominates Youtube ?

In [ ]:
full_df["trending_date"]

In [ ]:
full_df["trending_date"]= pd.to_datetime(full_df["trending_date"] , format = "%y.%d.%m")

In [ ]:
full_df.dtypes

In [ ]:
import json

In [ ]:
path = r'C:\Users\akash\Downloads\additional_data\US_category_id.json'

In [ ]:
with open(path, 'r' ,encoding ="utf-8")as f:
    data =json.load(f)

In [ ]:
data

In [ ]:
data["items"][0]

In [ ]:
data["items"][0]['snippet']['title']

In [ ]:
data["items"][0]['id']

In [ ]:
cat_dict = {}
for item in data["items"]:
    cat_dict[int(item['id'])] = item['snippet']['title']

In [ ]:
cat_dict

In [ ]:
full_df["category_name"]=full_df["category_id"].map(cat_dict)

In [ ]:
full_df.head(3)

In [ ]:
pivot_df = full_df.groupby(["trending_date", "category_name"])["views"].sum().unstack(fill_value = 0)

In [ ]:
pivot_df 


In [ ]:
area_chart = px.area(
    data_frame= pivot_df ,
    x = pivot_df.index ,
    y = pivot_df.columns ,
    title = "trending Momemntum over by category"
)

In [ ]:
area_chart

In [ ]:
top_categories = full_df.groupby(["category_name"])["views"].sum().nlargest(6).index

In [ ]:
top_categories

In [ ]:
filtered_df = pivot_df[top_categories]

In [ ]:
area_chart = px.area(
    data_frame=filtered_df   ,
    x = filtered_df .index ,
    y = filtered_df .columns ,
    title = "trending Momemntum over by category"
)

In [ ]:
area_chart

In [ ]:
 Do Viral Videos Actually Get Engagement ?

In [ ]:
full_df.columns

In [ ]:
full_df["engagement_rate"]= (full_df["likes"] + full_df["comment_count"]) / full_df["views"]

In [ ]:
bubble = px.scatter(full_df,
           x= "views",
           y= "engagement_rate",
           size = "comment_count",
           color ="category_name",
           hover_name ="title",
           title = "engegement Bubble Map : Views vs Engagement rate" ,
           size_max = 60)

In [ ]:
bubble.update_xaxes(type= 'log')

In [ ]:
 Views vs Engagement : Inside YouTube’s Algorithm !

In [ ]:
full_df.columns

In [ ]:
category_metrics = full_df.groupby('category_name').agg(
                                    total_views = ("views", "sum"),
                                    avg_engagement_efficiency = ("engagement_rate" , "mean"),
                                     video_count =("video_id", "count")).reset_index()


In [ ]:
category_metrics

In [ ]:
category_metrics.columns

In [ ]:
treemap = px.treemap(
                     category_metrics , 
                     path = ["category_name"] ,
                     values = "total_views" ,
                     color = "avg_engagement_efficiency" , 
                     color_continuous_scale = "RdYlGn" ,
                     title = "category attention  share with engagement efficiency overlay" , 
                     hover_data= {
                                 "total_views" : ":,.0f",
                                 'avg_engagement_efficiency' : ":.3f" ,
                                  "video_count" : True
                     }
)

In [ ]:
treemap

In [ ]:
 whether audience is engaged or not ! (descriptive analaysis)

In [ ]:
full_df.columns

In [ ]:
full_df["engagement_rate"].describe()

In [ ]:
category_engagement_stats = full_df.groupby('category_name')['engagement_rate'].describe()

In [ ]:
category_stats = category_engagement_stats.sort_values("mean" , ascending = False)

In [ ]:
category_stats

In [ ]:
full_df.columns

In [ ]:
box = px.box( full_df , 
       x = 'category_name',
       y = 'engagement_rate',
       color = 'category_name',
       title = "Audience engagement by category"
      )
       

In [ ]:
box
